# Auto Encoder
> https://excalidraw.com/#json=pWGS-CCIjK4BtIT32bs6F,T1KxjH7T1lPbDFhu8TnA7g
![image.png](image.png)

Given an input of $\text{input} \in \mathbb{R}^{D}$, one can represent the input data into smaller numbers or latents of size
$\text{latents} \in \mathbb{R}^{N}$ where $N \leq D$. Actually extremely simple with linear layers that progressively get smaller until we reach our 
desired latent, bottleneck size and then the reverse of linear layers that progressively project back to our original input size.

We can interpret the first part as basically "encoding" our data into $latent_size$ numbers that are then being decoded and 
reconstructed. 

In [ ]:
import torch
import torch.nn as nn
from dataclasses import dataclass

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
print(device)

cuda


In [79]:
from PIL import Image
import numpy as np

# number 7, 8x8 image
inputsize = 8 * 8

img = Image.open("test.png").convert("L")
arr = np.array(img, dtype=np.float32) / 255.0  # Normalize to [0, 1]

tensor = torch.from_numpy(arr).reshape(1, inputsize)

The linear sizes are hard coded. All you need to know is that the input was an $8x8$ image that was converted into a
size 64 vector so the $\text{input} \in \mathbb{R}^{64}$. Not a big fan of ReLU due to dead neurons but it's simple and fast.

Sigmoid at the end to return numbers for $0\sim1$ as it is the same format as black white images. Not sure if LayerNorm actually helps but my reasoning was to scale the numbers such that the variance fits with the sigmoid function as to have better gradients. Maybe residuals are better but probably Conv2d is the true better choice.

In [80]:
# hyperparams
@dataclass
class autoencConfig:
    inputsize: int = 64
    bottleneck: int = 8
    lr: float = 3e-4

class AutoEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.encode = nn.Sequential(
                nn.Linear(config.inputsize, 32),
                nn.ReLU(),
                nn.Linear(32, 16),
                nn.ReLU(),
                nn.Linear(16, 8),
                nn.ReLU()
        )
        
        self.bottleneck = nn.Linear(8, config.bottleneck)
        
        self.decode = nn.Sequential(
            nn.ReLU(),
            nn.Linear(config.bottleneck, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, config.inputsize)
        )
        
        self.ln = nn.LayerNorm(config.inputsize)
        
        
    def forward(self, x, targets=None):
        x = self.encode(x)
        x = self.bottleneck(x)
        x = self.decode(x)
        
        if targets is None:
            loss = None
        else:
            x = torch.sigmoid(self.ln(x))
            loss = nn.MSELoss()(x, targets)
        return x, loss

In [81]:
model = AutoEncoder(autoencConfig(inputsize=inputsize))
model.to(device=device)
optimizer = torch.optim.AdamW(model.parameters(), lr=model.config.lr)
outputs = []

In [82]:
iters = 250
for _ in range(iters):
    Xb = tensor.to(device=device)
    output, loss = model(Xb, Xb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    outputs.append(output)
    
print(loss.item())

import matplotlib.pyplot as plt
from ipywidgets import interact

@interact(i=(0, len(outputs)-1))
def show_image(i):
    arr = outputs[i].cpu().detach().reshape(8, 8).numpy()
    # img = Image.fromarray((arr * 255).clip(0, 255).astype(np.uint8), mode="L")
    # img.save("reconstructed.png")
    
    plt.figure(figsize=(5,5))
    plt.imshow(arr, cmap="gray", vmin=0, vmax=1)
    plt.title(f"Output {i}")
    plt.show()
    # plt.axis("off")

0.09556415677070618


interactive(children=(IntSlider(value=124, description='i', max=249), Output()), _dom_classes=('widget-interac…